# Milestone - Unit 6: grip and the speed limit

Unit 6 raced a car whose tyres can slide, and turned the tyres' ceiling into a speed limit.
`agents/unit-06.py` is Unit 5's file with one new constant, `SPEED_MODE`:
`"cap"` (the default: the speed goal is min(`TARGET_SPEED`, √(`A_LAT` / κ_ahead)), with κ_ahead Unit 2's curvature ahead over 25 waypoints, `A_LAT` = 6 m/s² and `TARGET_SPEED` = 20 m/s)
and `"flat"` (`TARGET_SPEED` everywhere: Unit 5's agent exactly).
Its pure pursuit, PI speed loop, instruments and model are Unit 5's.

The notebook asserts the unit's gates (plan `roar-006`) on the trimmed subset the plan names; a routed racekit test (`tests/test_racekit_unit06_07_premise.py`) races every gate in full:

- **U6.1** the hook, seed 0 on `test-circuit`: Unit 5's agent (`SPEED_MODE = "flat"`) spins 40 times on `DynamicCar()` and 39 times on `carla_like(noise=False)`, and finishes neither race in 1500 s;
- **U6.2** the cap against μg on `DynamicCar()`, `test-circuit`: `A_LAT` = 9.5 is clean in 410.05 s and 9.75 spins 36 times; the edge sits just under μg = 9.81 m/s²;
- **U6.4** the reference on `carla_like(noise=True)`, seed 0: clean with 0 spin-outs on every track, in the probe record's seed-0 times (216.50 / 422.55 / 880.40 s, plan roar-006); with `SPEED_MODE = "flat"` it races exactly like `agents/unit-05.py` on `KinematicCar()` (telemetry identical, and `roar-005`'s U5.2 times and RMS); `check_submission` passes.

Tolerances (the plan's Gates header): race times within ±0.05 s; clean/crash outcomes and spin counts exact.
A race is **clean** when it finishes with 0 collisions and its max |cross-track error| over the driving rows is at most 2 m.
Every figure is a model number on racekit's cars.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
A variant is the reference file with some constants changed (`write_variant`), written to a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import DynamicCar, KinematicCar, carla_like
from racekit.check_submission import check_submission
from racekit.runner import run_race

AGENTS = Path("..") / "agents"
AGENT = AGENTS / "unit-06.py"
UNIT_05 = AGENTS / "unit-05.py"
TRACKS = ("oval", "test-circuit", "monza-like")
TICK = 0.05 + 1e-9
RMS_TOL = 0.0005 + 1e-9
CLEAN_OFFSET = 2.0
workdir = Path(mkdtemp(prefix="roar-milestone-06-"))


def variant(source, name, **constants):
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def spins(result):
    return sum(cause == "spin" for cause in result.respawn_causes)


def max_offset(result):
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def rms_offset(result):
    telemetry = result.telemetry
    offset = telemetry["offset"][telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


def clean(result):
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


FLAT = variant(AGENT, "unit-06-flat", SPEED_MODE="flat")

## 2. U6.1: Unit 5's agent spins on the sliding cars

`test-circuit`, seed 0, `max_seconds=1500`.
Both races run to the cap (about 25 s together).

In [ ]:
HOOK = {"DynamicCar()": (DynamicCar, 40), "carla_like(noise=False)": (lambda: carla_like(noise=False), 39)}
for name, (make_car, spin_pin) in HOOK.items():
    result = run_race(FLAT, "test-circuit", car=make_car(), seed=0, max_seconds=1500.0)
    print(f"flat 20 m/s on {name:>24}: {result.collisions} collisions, {spins(result)} spin-outs, "
          f"finished {result.finished}")
    assert (spins(result), result.collisions, result.finished) == (spin_pin, spin_pin, False)

## 3. U6.2: the cap against μg

`DynamicCar()` (μ = 1, so μg = 9.81 m/s²), `test-circuit`, seed 0, `max_seconds=1500`: the two cells either side of the edge.

In [ ]:
below = run_race(variant(AGENT, "alat-9.5", A_LAT=9.5), "test-circuit", car=DynamicCar(), seed=0, max_seconds=1500.0)
above = run_race(variant(AGENT, "alat-9.75", A_LAT=9.75), "test-circuit", car=DynamicCar(), seed=0, max_seconds=1500.0)
print(f"A_LAT 9.5:  {below.total_time:.2f} s, {below.collisions} collisions, max |offset| {max_offset(below):.2f} m")
print(f"A_LAT 9.75: {above.collisions} collisions, {spins(above)} spin-outs, finished {above.finished}")
assert clean(below) and spins(below) == 0 and abs(below.total_time - 410.05) <= TICK
assert not above.finished and spins(above) == 36

## 4. U6.4: the reference

On `carla_like(noise=True)`, seed 0, `max_seconds=1500`, every track.
The pins are seed 0 of the plan's five-seed probe record (the routed test asserts the five-seed medians and spreads: 216.50 (0.10), 422.65 (0.30), 880.50 (0.15) s).

In [ ]:
SEED_0 = {"oval": 216.50, "test-circuit": 422.55, "monza-like": 880.40}
for track in TRACKS:
    result = run_race(AGENT, track, car=carla_like(noise=True), seed=0, max_seconds=1500.0)
    print(f"{track:>12}: {result.total_time:.2f} s, {result.collisions} collisions, {spins(result)} spin-outs, "
          f"max |offset| {max_offset(result):.2f} m")
    assert clean(result) and spins(result) == 0
    assert abs(result.total_time - SEED_0[track]) <= TICK

`SPEED_MODE = "flat"` must race exactly like `agents/unit-05.py`: on `KinematicCar()` (no jitter) the telemetry is compared column by column, and Unit 5's pinned U5.2 times and RMS hold.

In [ ]:
UNIT_05_PINNED = {"oval": (216.35, 0.0035), "test-circuit": (400.50, 0.0193), "monza-like": (858.45, 0.0157)}
for track in TRACKS:
    as_flat = run_race(FLAT, track, car=KinematicCar(jitter=0.0))
    unit_05 = run_race(UNIT_05, track, car=KinematicCar(jitter=0.0))
    assert len(as_flat.telemetry) == len(unit_05.telemetry)
    for column in unit_05.telemetry.dtype.names:  # identical telemetry, column by column
        assert np.max(np.abs(as_flat.telemetry[column] - unit_05.telemetry[column])) == 0, (track, column)
    time, rms = UNIT_05_PINNED[track]
    assert clean(as_flat) and abs(as_flat.total_time - time) <= TICK and abs(rms_offset(as_flat) - rms) <= RMS_TOL
    print(f"{track:>12}: SPEED_MODE = 'flat' {as_flat.total_time:.2f} s, RMS {rms_offset(as_flat):.4f} m, "
          "identical to agents/unit-05.py")

report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

## Unit 6, pinned

| on racekit's cars, `test-circuit` | Unit 5 (flat 20 m/s) | Unit 6 (the cap) |
|---|---|---|
| `DynamicCar()` (μ = 1) | 40 spin-outs, no finish | clean up to `A_LAT` 9.5 (410.05 s); 9.75 spins |
| `carla_like(noise=False)` | 39 spin-outs, no finish | clean up to `A_LAT` 7 (the routed test) |
| `carla_like(noise=True)`, the reference | | `A_LAT` 6: clean on every track and seed |

The submission ships the cap at `A_LAT = 6.0`, one grid step inside the carla-like car's clean edge (the margin rule), with Unit 5's speed one constant away (`SPEED_MODE = "flat"`, identical to the tick).
It is clean and slow: it never goes above 20 m/s, and it brakes only when a corner is inside its 50 m window.
Unit 7 plans the speed.